# GTFS-isochrone session info

Reproducibility footprint for the Auckland congestion charging health-equity pipeline. Captures Python, JVM, key Python package versions, and a quick check that the pipeline output GeoPackages are intact.

## Python and platform

In [1]:
import sys, platform
print("Python   :", sys.version)
print("Platform :", platform.platform())
print("Machine  :", platform.machine())
print("Processor:", platform.processor())

Python   : 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Platform : Linux-6.8.0-106-generic-aarch64-with-glibc2.35
Machine  : aarch64
Processor: aarch64


## JVM (required by r5py for the R5 routing engine)

In [2]:
import shutil, subprocess
java = shutil.which("java")
print("java path:", java)
if java:
    out = subprocess.run([java, "-version"], capture_output=True, text=True)
    print(out.stderr.strip() or out.stdout.strip())
else:
    print("java not found on PATH")

java path: /usr/bin/java
openjdk version "11.0.30" 2026-01-20
OpenJDK Runtime Environment (build 11.0.30+7-post-Ubuntu-1ubuntu122.04)
OpenJDK 64-Bit Server VM (build 11.0.30+7-post-Ubuntu-1ubuntu122.04, mixed mode)


## Key Python package versions

In [3]:
import importlib.metadata as m
pkgs = [
    "geopandas", "pandas", "numpy", "shapely", "pyproj", "fiona", "pyogrio",
    "matplotlib", "r5py", "gtfs_kit", "openpyxl", "pyarrow",
    "scipy", "scikit-learn",
]
for p in pkgs:
    try:
        print(f"{p:18} {m.version(p)}")
    except m.PackageNotFoundError:
        print(f"{p:18} (not installed)")

geopandas          1.1.3
pandas             2.3.3
numpy              2.2.6
shapely            2.1.2
pyproj             3.7.1
fiona              (not installed)
pyogrio            0.12.1
matplotlib         3.10.8
r5py               (not installed)
gtfs_kit           (not installed)
openpyxl           3.1.5
pyarrow            24.0.0
scipy              (not installed)
scikit-learn       (not installed)


## Pipeline data integrity

In [4]:
from pathlib import Path
import geopandas as gpd, pandas as pd

OUT = Path("../outputs")
DATA = Path("../data")

print("== outputs/ ==")
for p in sorted(OUT.glob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:40} {size_kb:>10.1f} KB")

print()
print("== data/ ==")
for p in sorted(DATA.glob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:40} {size_kb:>10.1f} KB")

== outputs/ ==
  .DS_Store                                       8.0 KB
  .fuse_hidden0000001300000001                 9048.4 KB
  .fuse_hidden0000001600000001                 8963.7 KB
  .fuse_hidden0000002600000002                 8992.6 KB
  .fuse_hidden0000002d00000003                 9040.7 KB
  .fuse_hidden0000003900000001                 8963.7 KB
  .fuse_hidden0000003c00000002                 8992.6 KB
  .fuse_hidden0000003c00000004                 9040.7 KB
  .fuse_hidden0000003e00000003                 9048.4 KB
  .fuse_hidden0000004200000004                 4999.4 KB
  burden_crosstab.csv                             1.0 KB
  equity_summary.csv                              0.3 KB
  sa2_accessibility.gpkg                      12900.0 KB
  sa2_equity.gpkg                             12932.0 KB
  sa2_equity_v2.gpkg                             96.0 KB
  sa2_final.gpkg                              12952.0 KB
  sa2_prepared.gpkg                           12884.0 KB
  sa2_prepared_f

In [5]:
sa2 = gpd.read_file("../outputs/sa2_equity.gpkg")
print(f"sa2_equity.gpkg rows: {len(sa2)}, cols: {len(sa2.columns)}")
print()
print("Burden columns and value counts:")
for col in [c for c in sa2.columns if c.startswith("burden_")]:
    counts = sa2[col].value_counts().to_dict()
    print(f"  {col}: {counts}")

sa2_equity.gpkg rows: 619, cols: 24

Burden columns and value counts:
  burden_1a: {'no_charge': 604, 'pays_with_alternative': 15}
  burden_1c: {'no_charge': 592, 'pays_with_alternative': 27}
  burden_2c: {'no_charge': 461, 'pays_with_alternative': 103, 'pays_without_alternative': 55}
  burden_3b: {'no_charge': 446, 'pays_without_alternative': 120, 'pays_with_alternative': 53}
  burden_3c: {'no_charge': 431, 'pays_without_alternative': 120, 'pays_with_alternative': 68}
  burden_3e: {'no_charge': 597, 'pays_without_alternative': 15, 'pays_with_alternative': 7}


## Equity summary (current pipeline run)

In [6]:
es = pd.read_csv("../outputs/equity_summary.csv")
print(es.to_string(index=False))

scenario  CI_charged_sa2s_45min  CI_no_alt_indicator  CI_all_30min  CI_all_45min  viable_alt_threshold_jobs
      1a                -0.0023                  NaN        0.1012        0.0421                    85782.0
      1c                 0.0282                  NaN        0.1012        0.0421                    85782.0
      2c                 0.0344              -0.0413        0.1012        0.0421                    85782.0
      3b                -0.2043               0.3686        0.1012        0.0421                    85782.0
      3c                -0.1445               0.3686        0.1012        0.0421                    85782.0
      3e                -0.0725               0.0406        0.1012        0.0421                    85782.0


In [7]:
bx = pd.read_csv("../outputs/burden_crosstab.csv")
print(bx.head(20).to_string(index=False))
print(f"\nTotal rows: {len(bx)}")

 NZDep_Decile  no_charge  pays_with_alternative scenario  pays_without_alternative
          1.0         61                      0       1a                       NaN
          2.0         61                      1       1a                       NaN
          3.0         62                      0       1a                       NaN
          4.0         64                      0       1a                       NaN
          5.0         57                      0       1a                       NaN
          6.0         61                      2       1a                       NaN
          7.0         56                      2       1a                       NaN
          8.0         58                      6       1a                       NaN
          9.0         55                      4       1a                       NaN
         10.0         60                      0       1a                       NaN
          1.0         60                      1       1c                       NaN
    

## Auckland-wide accessibility distribution (45 min)

In [8]:
print("Mean access_45min     :", int(sa2['access_45min'].mean()))
print("Median (Q50)          :", int(sa2['access_45min'].median()))
print("Q75 (cut-point)       :", int(sa2['access_45min'].quantile(0.75)))
print("Max                   :", int(sa2['access_45min'].max()))

Mean access_45min     : 64659
Median (Q50)          : 28730
Q75 (cut-point)       : 85782
Max                   : 355520


## `session_info` package output

In [9]:
import session_info
session_info.show(html=False, write_req_file=False)

-----
geopandas           1.1.3
pandas              2.3.3
session_info        v1.0.1
-----
IPython             8.39.0
jupyter_client      8.8.0
jupyter_core        5.9.1
jupyterlab          4.5.7
notebook            7.5.6
-----
Python 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Linux-6.8.0-106-generic-aarch64-with-glibc2.35
-----
Session information updated at 2026-05-02 09:32


---
*Generated to capture the analysis environment for the City Shorts paper draft.*